# 01 - Exploration visuelle & préparation des données
**Branche** : `feature/01-data-exploration`

**Objectifs** : vue d'ensemble du jeu, qualité (corruption, doublons, outliers), homogénéité des sources, nettoyage traçable.

> Hypothèse d'arborescence : `data/raw/labeled/{normal,cancer}/` et `data/raw/unlabeled/`.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from brainscanai.config import load_config, resolve_path
from brainscanai.utils import set_seed

cfg = load_config()
set_seed(cfg["seed"])
PROC, FEAT = resolve_path(cfg, "processed", create=True), resolve_path(cfg, "features", create=True)
FIG, TAB = resolve_path(cfg, "figures", create=True), resolve_path(cfg, "tables", create=True)
from brainscanai.data.loader import build_manifest
from brainscanai.data.quality import *
from brainscanai.evaluation.plots import plot_sample_grid, save_fig

## 1. Manifeste & structure

In [2]:
manifest = build_manifest(cfg)
print(manifest.shape)
manifest.groupby(['source', 'label_name']).size().rename('n_images').to_frame()

(1506, 6)


n_images
source    label_name          
labeled   cancer            50
          normal            50
unlabeled unlabeled       1406

## 2. Exemples d'images (par classe)

In [3]:
rng = np.random.default_rng(cfg['seed'])
for name in ['normal', 'cancer', 'unlabeled']:
    sub = manifest[manifest.label_name == name]
    pick = sub.sample(min(5, len(sub)), random_state=cfg['seed'])
    fig = plot_sample_grid(pick.path.tolist(), [name] * len(pick))
    fig.suptitle(name, y=1.02); save_fig(fig, f'samples_{name}', FIG); plt.show()

## 3. Résolution, canaux, format, intensité, netteté
Le fichier de description annonce 512×512 JPEG ; l'énoncé de la mission parle de PNG : **on vérifie plutôt que de supposer**.

In [4]:
stats = compute_image_stats(manifest)
print(stats[['width', 'height', 'mode', 'format']].value_counts(dropna=False))
stats[['mean', 'std', 'sharpness', 'black_ratio']].describe().round(2)

width  height  mode  format
512    512     RGB   JPEG      1506
Name: count, dtype: int64


,mean,std,sharpness,black_ratio
count,1506.00,1506.00,1506.00,1506.00
mean,52.40,48.94,117.79,0.30
std,17.38,13.71,356.84,0.16
min,14.95,20.89,8.06,0.00
25%,41.29,39.91,38.31,0.18
50%,50.26,45.03,54.75,0.33
75%,59.25,54.31,79.55,0.40
max,136.22,119.18,5789.08,0.75


## 4. Outliers, doublons, fichiers corrompus
Z-score robuste (médiane/MAD). **Règle** : jamais de suppression d'une image *labellisée* (signalement seulement) ; doublons : on garde la copie labellisée.

In [5]:
q = cfg['data']['quality']
stats = flag_duplicates(flag_outliers(stats, z=q['outlier_robust_z']).assign(md5=lambda d: d['md5'].fillna('x_' + d['image_id'].astype(str))))
print('corrompues :', int(stats.corrupt.sum()), '| doublons :', int(stats.is_duplicate.sum()), '| outliers :', int(stats.is_outlier.sum()))
out = stats[stats.is_outlier | stats.is_duplicate]
out.head(10)

corrompues : 0 | doublons : 96 | outliers : 144


,image_id,source,label,corrupt,md5,width,height,mode,format,mean,std,min,max,sharpness,black_ratio,outlier_reason,is_outlier,is_duplicate
2,2,labeled,0,False,6d7b20c66c53b3ca21faf45932d27169,512,512,RGB,JPEG,98.114876,90.388206,0.0,255.0,105.572395,0.353008,std;,True,False
6,6,labeled,0,False,eaeb102cd99661f157923b9eff689053,512,512,RGB,JPEG,53.585484,58.988605,0.0,255.0,171.127731,0.013054,sharpness;,True,False
7,7,labeled,0,False,df7e098b4eb81f5b81865c93ead0ccd1,512,512,RGB,JPEG,93.675156,87.977547,0.0,255.0,2051.494629,0.090088,std;sharpness;,True,False
9,9,labeled,0,False,637d62b900ddf7e7632146f0caa821a2,512,512,RGB,JPEG,36.340221,59.696991,0.0,255.0,221.386444,0.632740,sharpness;,True,False
11,11,labeled,0,False,d4b8493bf3e24910bdad482dc7502a77,512,512,RGB,JPEG,62.872204,63.825584,0.0,255.0,275.553986,0.328087,sharpness;,True,False
13,13,labeled,0,False,234bef5f52771afb417f07b260c261f9,512,512,RGB,JPEG,89.677322,85.391235,0.0,255.0,136.437622,0.207409,std;,True,False
20,20,labeled,0,False,427d44a774938b994e209142d4f31291,512,512,RGB,JPEG,100.580475,73.931320,0.0,255.0,1490.354980,0.061089,sharpness;,True,False
24,24,labeled,0,False,c33d9bfe54114ba89a9aefbd2f913e84,512,512,RGB,JPEG,98.263893,90.291115,0.0,255.0,117.320709,0.339718,std;,True,False
25,25,labeled,0,False,9fc79860dc2892183230155ffe0965d0,512,512,RGB,JPEG,60.053272,76.659660,0.0,255.0,255.857803,0.509293,sharpness;,True,False
26,26,labeled,0,False,4f5f80f9074b2997d8dff6aaec9771f8,512,512,RGB,JPEG,115.188698,86.314980,0.0,255.0,256.949554,0.013336,mean;std;sharpness;,True,False


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, col in zip(axes, ['mean', 'std', 'sharpness']):
    for src, g in stats.groupby('source'):
        ax.hist(g[col].dropna(), bins=30, alpha=.6, density=True, label=src)
    ax.set_title(col); ax.legend()
save_fig(fig, 'intensity_by_source', FIG); plt.show()

## 5. Biais potentiels : labellisées vs non labellisées (test de Kolmogorov-Smirnov)
Un décalage de distribution (scanner, protocole) fausserait la propagation des labels.

In [7]:
domain_shift_report(stats)

,feature,ks_stat,p_value,shift_suspected
0,mean,0.098421,0.304818,False
1,std,0.153670,0.021567,False
2,sharpness,0.258634,0.000005,True


## 6. Nettoyage & sauvegarde

In [8]:
cleaned, removed = clean_manifest(manifest, stats, q['drop_duplicates'], q['drop_unlabeled_outliers'])
print(f'{len(manifest)} -> {len(cleaned)} images ; retirées : {len(removed)}')
stats.to_csv(PROC / 'image_stats.csv', index=False)
cleaned.to_csv(PROC / 'manifest_clean.csv', index=False)
removed.to_csv(PROC / 'removed_images.csv', index=False)
removed.reason.value_counts()

1506 -> 1305 images ; retirées : 201


reason
outlier;              105
duplicate;             77
duplicate;outlier;     19
Name: count, dtype: int64

## Observations

Le jeu compte 1 506 images au total : 100 labellisées, parfaitement équilibrées (50 cancer et 50 normal), et 1 406 non labellisées. Le volume annoté étant très faible, le recours au semi-supervisé est justifié a priori ; son apport réel sera mesuré au notebook 04.

- Format et intégrité : tout est uniforme (512×512, RGB, JPEG) et aucun fichier n'est corrompu. L'énoncé parlait de PNG, mais la vérification confirme que 100 % du dataset est bien en JPEG.

- Qualité visuelle et outliers : la proportion de fond noir (black_ratio, jusqu’à 0,75) et la netteté (médiane autour de 55, mais valeurs extrêmes jusqu’à 5 789) varient fortement. Les causes de ces extrêmes (coupes peu informatives, artefacts, contraste élevé) restent à confirmer visuellement. Côté non labellisé, 201 images ont été retirées (96 doublons MD5 et 124 outliers, dont 19 cumulent les deux), ce qui laisse 1 205 images. N° images labellisées sont signalées comme outliers : elles sont conservées, mais leur comportement sera surveillé en validation. Le critère d’outlier étant sensible aux distributions asymétriques et au contraste d’une tumeur, leur répartition par classe a été contrôlée pour s’assurer qu’il ne retire pas de cas pathologiques.

- Écart labellisé / non labellisé : le test KS ne montre pas de biais de luminosité (p = 0,30). L’écart de contraste (p = 0,02) n’est pas significatif après correction pour trois tests, tandis que celui de netteté (p = 5 × 10⁻⁶) l’est nettement. Ce test est à refaire après nettoyage, [et à vérifier entre classes : si la netteté diffère aussi entre cancer et normal, le modèle risque d’apprendre l’origine de l’image plutôt que la tumeur]. L’augmentation de données (flou, bruit, contraste) n’est envisagée qu’après ce contrôle.